In [1]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/TwinCheck'
REPO = '/content/Group-Project'
OUT = f'{DRIVE}/results/F3'
!rm -rf {REPO}
!git clone -q --depth 1 https://github.com/himaallu/Group-Project.git {REPO}
!pip -q install lightgbm
!nvidia-smi --query-gpu=name,memory.total --format=csv

Mounted at /content/drive
name, memory.total [MiB]
Tesla T4, 15360 MiB


In [2]:
import io
import os
import subprocess
import sys
from concurrent.futures import ThreadPoolExecutor
from glob import glob

import numpy as np
import pandas as pd
import torch
from PIL import Image
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

os.chdir(REPO)
sys.path.insert(0, REPO)
SEED = 42
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
os.makedirs(f'{OUT}/baselines', exist_ok=True)

gen = pd.concat([pd.read_csv(f'data/processed/vision/genimage_{s}_manifest.csv') for s in ('train', 'validation', 'test')], ignore_index=True)
gen = gen.merge(pd.read_csv('features/vision/genimage/genimage_embedding_manifest.csv')[['embedding_row', 'md5']], on='md5', how='left', validate='one_to_one')
gen['y'] = (gen['class_name'] == 'fake').astype(int)
verite = pd.read_csv('data/processed/vision/verite_manifest.csv')
twins = pd.read_csv('data/copies/vision/verite_vae_twins.csv')
twin_rows = pd.read_csv('features/vision/verite/verite_twin_embedding_manifest.csv')
captions = pd.read_csv('features/vision/verite/verite_caption_embedding_manifest.csv')
sdxl = pd.read_csv('data/copies/vision/sdxl_twins.csv')
sdxl_real = sdxl[sdxl['class_name'] == 'real'].reset_index(drop=True)
NPY = {
    'genimage_clip': 'features/vision/genimage/clip_vitl14/embeddings_normalized.npy',
    'verite_image_siglip': 'features/vision/verite/siglip2_so400m/embeddings_normalized.npy',
    'verite_caption_siglip': 'features/vision/verite/siglip2_text/embeddings_normalized.npy',
    'verite_twin_siglip': 'features/vision/verite/siglip2_vae_twins/embeddings_normalized.npy',
}

assert len(gen) == 13000 and gen['embedding_row'].notna().all()
assert gen.groupby(['split', 'generator']).size().to_dict() == {
    ('test', 'Stable Diffusion v1.4'): 1000, ('test', 'real'): 1000,
    ('train', 'BigGAN'): 5000, ('train', 'real'): 4000,
    ('validation', 'BigGAN'): 1000, ('validation', 'real'): 1000}
assert len(twins) == 218


def missing_on_drive(paths):
    listing, missing = {}, []
    for p in paths:
        folder, name = os.path.split(f'{DRIVE}/{p}')
        if folder not in listing:
            listing[folder] = set(os.listdir(folder)) if os.path.isdir(folder) else set()
        if name not in listing[folder]:
            missing.append(p)
    return missing


needed = [*NPY.values(), *gen['raw_export_path'], *twins['absolute_path'], *twins['twin_path'], *sdxl_real['final_twin_path_jpeg85']]
missing = missing_on_drive(needed)
print('Drive files needed:', len(needed), '| missing:', len(missing))
for p in missing[:20]:
    print('  missing:', p)
assert not missing, 'Upload the missing files to MyDrive/TwinCheck with the same folder layout'
assert DEVICE == 'cuda', 'Runtime > Change runtime type > T4 GPU'
print('device:', DEVICE)

Drive files needed: 13631 | missing: 0
device: cuda


In [3]:
import hashlib

from PIL import ImageOps, JpegImagePlugin
from torchvision import transforms
from transformers import CLIPVisionModelWithProjection

SIZE, BATCH, POOL_SIZE = 96, 128, 500
CLIP_ID = 'openai/clip-vit-large-patch14'
FEAT = f'{DRIVE}/features/vision/genimage/clip_vitl14_debiased{SIZE}'


def jpeg_settings(path):
    with Image.open(path) as im:
        return [list(t) for t in im.quantization.values()], JpegImagePlugin.get_sampling(im)


def load_square(path):
    with Image.open(path) as im:
        im = ImageOps.exif_transpose(im).convert('RGB')
        w, h = im.size
        s = min(w, h)
        left, top = (w - s) // 2, (h - s) // 2
        return np.asarray(im.crop((left, top, left + s, top + s)).resize((SIZE, SIZE), Image.Resampling.BICUBIC))


def load_all(paths):
    with ThreadPoolExecutor(32) as pool:
        return np.stack(list(pool.map(lambda p: load_square(f'{DRIVE}/{p}'), paths)))


def chroma(a):
    return np.abs(np.diff(a.astype(np.int16), axis=-1)).mean(axis=tuple(range(1, a.ndim)))


def encode(a, key):
    tables, sampling = JPEG_POOL[int(hashlib.md5(key.encode()).hexdigest()[:8], 16) % len(JPEG_POOL)]
    buf = io.BytesIO()
    Image.fromarray(a).save(buf, format='JPEG', qtables=tables, subsampling=sampling)
    return buf.getvalue()


def encode_all(arrays, keys):
    with ThreadPoolExecutor(8) as pool:
        return list(pool.map(encode, arrays, keys))


def process(paths):
    return encode_all(load_all(paths), list(paths))


pool_rows = gen[(gen['split'] == 'train') & (gen['class_name'] == 'real') & (gen['original_mode'] == 'RGB')].sample(POOL_SIZE, random_state=SEED)
with ThreadPoolExecutor(32) as pool:
    settings = list(pool.map(lambda p: jpeg_settings(f'{DRIVE}/{p}'), pool_rows['raw_export_path']))
JPEG_POOL = [s for s in settings if len(s[0]) >= 2 and s[1] in (0, 1, 2)]
assert JPEG_POOL
print('JPEG settings taken from real training photos:', len(JPEG_POOL))

pixels = load_all(gen['raw_export_path'])
gen['short_side'] = gen[['original_width', 'original_height']].min(axis=1)
gen['grayscale_original'] = chroma(pixels) < 1.0
gen['grayscale_added'] = False
rng = np.random.default_rng(SEED)
for split in ('train', 'validation', 'test'):
    rate = {c: gen.loc[(gen['split'] == split) & (gen['class_name'] == c), 'grayscale_original'].mean() for c in ('real', 'fake')}
    high = max(rate, key=rate.get)
    low = 'fake' if high == 'real' else 'real'
    rows = (gen['split'] == split) & (gen['class_name'] == low)
    n = int(round(rate[high] * rows.sum())) - int(gen.loc[rows, 'grayscale_original'].sum())
    if n > 0:
        gen.loc[rng.choice(gen.index[rows & ~gen['grayscale_original']], n, replace=False), 'grayscale_added'] = True
for i in np.flatnonzero(gen['grayscale_added']):
    pixels[i] = np.asarray(Image.fromarray(pixels[i]).convert('L').convert('RGB'))
gen['grayscale'] = gen['grayscale_original'] | gen['grayscale_added']
print(gen.groupby(['split', 'class_name']).agg(images=('md5', 'size'), grayscale_before=('grayscale_original', 'mean'), grayscale_after=('grayscale', 'mean'),
                                              upscaled=('short_side', lambda s: int((s < SIZE).sum()))).round(4).to_string())

gen_blobs = encode_all(pixels, gen['md5'])
del pixels

clip = CLIPVisionModelWithProjection.from_pretrained(CLIP_ID, torch_dtype=torch.float16 if DEVICE == 'cuda' else torch.float32).to(DEVICE).eval()
to_tensor = transforms.Compose([
    transforms.Resize(224, interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.48145466, 0.4578275, 0.40821073], [0.26862954, 0.26130258, 0.27577711]),
])


@torch.inference_mode()
def embed(blobs):
    out = []
    for i in range(0, len(blobs), BATCH):
        x = torch.stack([to_tensor(Image.open(io.BytesIO(b)).convert('RGB')) for b in blobs[i:i + BATCH]])
        e = clip(pixel_values=x.to(DEVICE, clip.dtype)).image_embeds.float()
        out.append(torch.nn.functional.normalize(e, dim=1).cpu().numpy())
    return np.concatenate(out)


X_input = embed(gen_blobs)
X_f2 = np.load(f'{DRIVE}/{NPY["genimage_clip"]}', mmap_mode='r')[gen['embedding_row'].astype(int).to_numpy()].astype(np.float32)
verite_orig_clip = embed(process(twins['absolute_path']))
verite_twin_clip = embed(process(twins['twin_path']))
sdxl_edit_clip = embed(process(sdxl_real['final_twin_path_jpeg85']))

os.makedirs(FEAT, exist_ok=True)
np.save(f'{FEAT}/embeddings_normalized.npy', X_input.astype(np.float16))
gen[['md5', 'split', 'class_name', 'generator', 'original_format', 'short_side', 'grayscale_original', 'grayscale_added']].to_csv(f'{FEAT}/embedding_manifest.csv', index=False)
del clip
torch.cuda.empty_cache()
print('detector embeddings', X_input.shape, '| F2 embeddings', X_f2.shape, '| VERITE', verite_orig_clip.shape, verite_twin_clip.shape, '| SDXL edits', sdxl_edit_clip.shape)

JPEG settings taken from real training photos: 500
                       images  grayscale_before  grayscale_after  upscaled
split      class_name                                                     
test       fake          1000            0.0210           0.0210         0
           real          1000            0.0150           0.0210         9
train      fake          5000            0.0214           0.0214         0
           real          4000            0.0135           0.0215        22
validation fake          1000            0.0230           0.0230         0
           real          1000            0.0210           0.0230         3


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/4.52k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.71GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/392 [00:00<?, ?it/s]

[transformers] CLIPVisionModelWithProjection LOAD REPORT from: openai/clip-vit-large-patch14
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
text_projection.weight                                       | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{

detector embeddings (13000, 768) | F2 embeddings (13000, 768) | VERITE (218, 768) (218, 768) | SDXL edits (191, 768)


In [4]:
def input_cues(blobs):
    rows = []
    for b in blobs:
        a = np.asarray(Image.open(io.BytesIO(b)).convert('RGB'), dtype=np.int16)
        g = a.mean(axis=-1)
        dx, dy = np.abs(np.diff(g, axis=1)), np.abs(np.diff(g, axis=0))
        block = (dx[:, 7::8].mean() / (dx.mean() + 1e-6) + dy[7::8].mean() / (dy.mean() + 1e-6)) / 2
        rows.append([chroma(a[None])[0], block])
    return np.array(rows)


masks = {s: (gen['split'] == s).to_numpy() for s in ('train', 'validation', 'test')}
y = gen['y'].to_numpy()
meta = np.c_[(gen['original_format'] == 'PNG').astype(float), np.log(gen['original_width']), np.log(gen['original_height'])]
cues = input_cues(gen_blobs)
DETECTOR = f'debiased {SIZE} px CLIP embeddings (detector)'
INPUTS = {
    'original file format and size (never given to the detector)': meta,
    'F2 embeddings (256 padded copies)': X_f2,
    'detector input cues (grayscale, JPEG blockiness)': cues,
    DETECTOR: X_input,
}


def make_model(kind, X):
    if kind == 'logistic regression':
        return make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, random_state=SEED)) if X.shape[1] < 10 else LogisticRegression(max_iter=2000, random_state=SEED)
    return LinearSVC(C=1.0, max_iter=10000, random_state=SEED)


def score(model, X):
    return model.predict_proba(X)[:, 1] if hasattr(model, 'predict_proba') else model.decision_function(X)


def best_threshold(y_true, s):
    order = np.argsort(-s)
    tp = np.cumsum(y_true[order])
    fp = np.cumsum(1 - y_true[order])
    f1 = 2 * tp / (tp + fp + y_true.sum())
    return float(s[order][np.argmax(f1)])


def metric_row(y_true, s, thr):
    p = (s >= thr).astype(int)
    return {'accuracy': accuracy_score(y_true, p), 'balanced_accuracy': balanced_accuracy_score(y_true, p),
            'precision': precision_score(y_true, p, zero_division=0), 'recall': recall_score(y_true, p, zero_division=0),
            'f1': f1_score(y_true, p, zero_division=0), 'auroc': roc_auc_score(y_true, s),
            'real_flagged_as_ai': float(p[y_true == 0].mean())}


rows, probes = [], {}
for name, X in INPUTS.items():
    for kind in (['logistic regression'] if X.shape[1] < 10 else ['logistic regression', 'linear SVM']):
        model = make_model(kind, X).fit(X[masks['train']], y[masks['train']])
        probes[(name, kind)] = model
        default = 0.5 if kind == 'logistic regression' else 0.0
        tuned = best_threshold(y[masks['validation']], score(model, X[masks['validation']]))
        for split, label in (('validation', 'seen generator (BigGAN)'), ('test', 'unseen generator (SD v1.4)')):
            s = score(model, X[masks[split]])
            for rule, thr in (('default', default), ('validation-tuned', tuned)):
                rows.append({'input': name, 'model': kind, 'split': label, 'threshold_rule': rule, 'threshold': thr,
                             **metric_row(y[masks[split]], s, thr)})
probe_results = pd.DataFrame(rows)
probe_results.to_csv(f'{OUT}/F3_probe_results.csv', index=False)
pd.set_option('display.width', 250)
print(probe_results[probe_results['threshold_rule'] == 'default'].drop(columns=['threshold_rule', 'threshold']).round(3).to_string(index=False))

                                                      input               model                      split  accuracy  balanced_accuracy  precision  recall    f1  auroc  real_flagged_as_ai
original file format and size (never given to the detector) logistic regression    seen generator (BigGAN)     1.000              1.000      1.000   1.000 1.000  1.000               0.000
original file format and size (never given to the detector) logistic regression unseen generator (SD v1.4)     1.000              1.000      1.000   1.000 1.000  1.000               0.000
                          F2 embeddings (256 padded copies) logistic regression    seen generator (BigGAN)     0.995              0.995      0.991   0.999 0.995  1.000               0.009
                          F2 embeddings (256 padded copies) logistic regression unseen generator (SD v1.4)     0.496              0.496      0.182   0.002 0.004  0.536               0.009
                          F2 embeddings (256 padded copies) 

In [5]:
auth_probe = probes[(DETECTOR, 'logistic regression')]
p_input = auth_probe.predict_proba(X_input)[:, 1]
groups = {
    'GenImage real photos (test)': p_input[masks['test'] & (y == 0)],
    'GenImage BigGAN (validation, seen)': p_input[masks['validation'] & (y == 1)],
    'GenImage SD v1.4 (test, unseen)': p_input[masks['test'] & (y == 1)],
    'SDXL img2img edits of real test photos': auth_probe.predict_proba(sdxl_edit_clip)[:, 1],
    'VERITE news photos (originals)': auth_probe.predict_proba(verite_orig_clip)[:, 1],
    'VERITE VAE twins': auth_probe.predict_proba(verite_twin_clip)[:, 1],
}
camera_ai_edited = pd.DataFrame([{'group': g, 'images': len(v), 'mean_p_ai': v.mean(), 'median_p_ai': np.median(v),
                                  'flagged_as_ai_at_0.5': (v >= 0.5).mean()} for g, v in groups.items()])
camera_ai_edited.to_csv(f'{OUT}/F3_camera_ai_edited.csv', index=False)
print(camera_ai_edited.round(3).to_string(index=False))

auth = pd.DataFrame({'image_file': twins['original_filename'],
                     'original_authenticity_score': groups['VERITE news photos (originals)'],
                     'twin_authenticity_score': groups['VERITE VAE twins']})
auth['flag_changed'] = (auth['original_authenticity_score'] >= 0.5) != (auth['twin_authenticity_score'] >= 0.5)
auth.to_csv(f'{OUT}/F3_verite_authenticity_scores.csv', index=False)
print('VERITE twins whose authenticity flag changes:', int(auth['flag_changed'].sum()), 'of', len(auth))

                                 group  images  mean_p_ai  median_p_ai  flagged_as_ai_at_0.5
           GenImage real photos (test)    1000      0.137        0.057                 0.072
    GenImage BigGAN (validation, seen)    1000      0.887        0.930                 0.980
       GenImage SD v1.4 (test, unseen)    1000      0.218        0.112                 0.156
SDXL img2img edits of real test photos     191      0.297        0.167                 0.246
        VERITE news photos (originals)     218      0.085        0.036                 0.018
                      VERITE VAE twins     218      0.103        0.044                 0.032
VERITE twins whose authenticity flag changes: 5 of 218


In [6]:
S_img = np.load(f'{DRIVE}/{NPY["verite_image_siglip"]}')
S_cap = np.load(f'{DRIVE}/{NPY["verite_caption_siglip"]}')
S_twin = np.load(f'{DRIVE}/{NPY["verite_twin_siglip"]}')
assert len(S_img) == 664 and len(S_cap) == len(captions) == 1001 and len(S_twin) == len(twin_rows) == 664

split_of = {os.path.basename(p): s for p, s in zip(verite['image_path'], verite['split'])}
twin_row_of = dict(zip(twin_rows['original_embedding_index'], twin_rows['embedding_row']))
twin_path_of = dict(zip(twin_rows['original_embedding_index'], twin_rows['twin_path']))
img = twins[['original_filename', 'original_embedding_index', 'twin_path']].rename(columns={'original_filename': 'image_file'})
assert all(twin_path_of[i] == p for i, p in zip(img['original_embedding_index'], img['twin_path']))
img['twin_row'] = img['original_embedding_index'].map(twin_row_of)
img['split'] = img['image_file'].map(split_of)
img['clip_index'] = np.arange(len(img))
img = img.merge(auth[['image_file', 'original_authenticity_score', 'twin_authenticity_score']], on='image_file')

cap = captions[captions['label'].isin(['true', 'miscaptioned'])][['embedding_index', 'image_filename', 'label']]
cap = cap.rename(columns={'image_filename': 'image_file'}).merge(img, on='image_file', how='inner')
assert (cap.groupby('image_file')['label'].apply(sorted).apply(tuple) == ('miscaptioned', 'true')).all()
assert cap['split'].notna().all()
cap['y'] = (cap['label'] == 'true').astype(int)
cap['sim_original'] = (S_img[cap['original_embedding_index']] * S_cap[cap['embedding_index']]).sum(1)
cap['sim_twin'] = (S_twin[cap['twin_row']] * S_cap[cap['embedding_index']]).sum(1)

train = cap['split'] == 'train'
pca = PCA(n_components=16, random_state=SEED).fit(verite_orig_clip[img.loc[img['split'] == 'train', 'clip_index']])
E = {'original': pca.transform(verite_orig_clip[cap['clip_index']]), 'twin': pca.transform(verite_twin_clip[cap['clip_index']])}


def features(model, version):
    sim, a = cap[f'sim_{version}'].to_numpy(), cap[f'{version}_authenticity_score'].to_numpy()
    if model == 'decoupled':
        return sim[:, None]
    base = np.c_[sim, a, sim * a]
    return base if model == 'fused' else np.c_[base, E[version]]


MODELS = ('decoupled', 'fused', 'fused + image embedding')
fitted = {m: make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, random_state=SEED)).fit(features(m, 'original')[train], cap.loc[train, 'y']) for m in MODELS}

pair_rows, caption_rows = [], []
for m in MODELS:
    for version in ('original', 'twin'):
        prob = fitted[m].predict_proba(features(m, version))[:, 1]
        part = cap.assign(prob=prob)
        for split in ('validation', 'test'):
            s = part[part['split'] == split]
            caption_rows.append({'model': m, 'version': version, 'split': split, 'captions': len(s),
                                 'accuracy': accuracy_score(s['y'], (s['prob'] >= 0.5).astype(int))})
        piv = part.pivot(index='image_file', columns='label', values='prob')
        for f, r in piv.iterrows():
            pair_rows.append({'model': m, 'version': version, 'image_file': f, 'split': split_of[f],
                              'p_true_caption': r['true'], 'p_miscaptioned': r['miscaptioned'],
                              'picks_true_caption': bool(r['true'] > r['miscaptioned'])})
pairs = pd.DataFrame(pair_rows)
pairs.to_csv(f'{OUT}/F3_pair_verdicts.csv', index=False)
pd.DataFrame(caption_rows).to_csv(f'{OUT}/F3_caption_metrics.csv', index=False)

wide = pairs.pivot_table(index=['model', 'image_file', 'split'], columns='version', values='picks_true_caption').reset_index()
summary = []
for (m, split), g in wide[wide['split'].isin(['validation', 'test'])].groupby(['model', 'split']):
    changed = int((g['original'] != g['twin']).sum())
    summary.append({'model': m, 'split': split, 'pairs': len(g), 'original_accuracy': g['original'].mean(),
                    'twin_accuracy': g['twin'].mean(), 'changed_verdicts': changed, 'ALR_percent': 100 * changed / len(g)})
leakage = pd.DataFrame(summary).sort_values(['split', 'model'])
leakage.to_csv(f'{OUT}/F3_leakage_ALR.csv', index=False)
print('pairs per split:', img['split'].value_counts().to_dict())
print(leakage.round(3).to_string(index=False))

pairs per split: {'train': 150, 'test': 36, 'validation': 32}
                  model      split  pairs  original_accuracy  twin_accuracy  changed_verdicts  ALR_percent
              decoupled       test     36              0.528          0.528                 4       11.111
                  fused       test     36              0.556          0.556                 8       22.222
fused + image embedding       test     36              0.556          0.528                 9       25.000
              decoupled validation     32              0.812          0.719                 3        9.375
                  fused validation     32              0.781          0.688                 5       15.625
fused + image embedding validation     32              0.781          0.688                 3        9.375


In [7]:
EN = 'data/processed/en'
BASE = f'{OUT}/baselines'
common = ['--train', f'{EN}/multifc_train.jsonl', '--val', f'{EN}/multifc_val.jsonl',
          '--predict', f'{EN}/multifc_test.jsonl', f'{EN}/averitec_test.jsonl', '--no-wandb']
for p in glob(f'{BASE}/preds_*.jsonl'):
    os.remove(p)
subprocess.run([sys.executable, '-m', 'src.eval.baselines.claim_only', *common, '--out', f'{BASE}/preds_claim_only.jsonl'], check=True)
subprocess.run([sys.executable, '-m', 'src.eval.baselines.similarity', *common, '--out-dir', BASE], check=True)
subprocess.run([sys.executable, '-m', 'src.eval.baselines.audit', '--preds', *sorted(glob(f'{BASE}/preds_*.jsonl')),
                '--gold', f'{EN}/multifc_test.jsonl', f'{EN}/averitec_test.jsonl', '--out', f'{OUT}/baselines_originals.csv'], check=True)
baselines = pd.read_csv(f'{OUT}/baselines_originals.csv')
print(baselines[['model', 'dataset', 'n_seeds', 'n_orig', 'macro_f1', 'macro_f1_sd', 'auroc']].round(3).to_string(index=False))

     model  dataset  n_seeds  n_orig  macro_f1  macro_f1_sd  auroc
claim_only averitec        1     429     0.334          NaN  0.522
claim_only  multifc        1    2106     0.482          NaN  0.719
similarity averitec        3     429     0.281        0.007  0.454
similarity  multifc        3    2106     0.385        0.005  0.609


In [8]:
default = probe_results[probe_results['threshold_rule'] == 'default']
summary = pd.concat([
    default.assign(part='authenticity probe')[['part', 'input', 'model', 'split', 'accuracy', 'recall', 'real_flagged_as_ai', 'auroc']],
    leakage.assign(part='VERITE leakage')[['part', 'model', 'split', 'original_accuracy', 'twin_accuracy', 'ALR_percent']],
    baselines.assign(part='English baselines')[['part', 'model', 'dataset', 'macro_f1', 'auroc']],
], ignore_index=True)
summary.to_csv(f'{OUT}/F3_summary.csv', index=False)
for f in sorted(os.listdir(OUT)):
    print(f)

F3_camera_ai_edited.csv
F3_caption_metrics.csv
F3_leakage_ALR.csv
F3_pair_verdicts.csv
F3_probe_results.csv
F3_summary.csv
F3_verite_authenticity_scores.csv
baselines
baselines_originals.csv
